# Setlist Generator

Generate a Phish setlist with configurable parameters.

In [1]:
from datetime import date
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from phish_setlist_maker.db import session_scope
from phish_setlist_maker.generator import SetlistGenerator, random_set_lengths
from phish_setlist_maker.models import Show

with session_scope() as session:
    latest_show = session.query(Show).order_by(Show.date.desc()).first()
    latest = latest_show.date if latest_show else 'N/A'
    print(f'Latest show in database: {latest}')


Latest show in database: 2024-10-27


In [ ]:
# Configuration
reference_date = None  # Optionally anchor generation to a specific date
era = '3.0'            # Use None for all history or choose '1.0', '2.0', '3.0', '4.0'
year = 2018            # Restrict history to shows through this calendar year
num_sets = 2           # Standard Phish format: 2 or 3 sets
include_encore = True  # Include an encore segment
random_seed = None     # Set to an int for reproducible set-length sampling


In [ ]:
# Sample set lengths from historical distributions
from random import Random

rng = Random(random_seed) if random_seed is not None else None
with session_scope() as session:
    set_lengths = random_set_lengths(
        session,
        reference_date=reference_date,
        era=era,
        year=year,
        num_sets=num_sets,
        include_encore=include_encore,
        rng=rng,
    )

set_lengths


In [3]:
# Generate setlist
with session_scope() as session:
    generator = SetlistGenerator(session)
    generated = generator.generate(
        reference_date=reference_date,
        era=era,
        year=year,
        num_sets=num_sets,
        include_encore=include_encore,
        set_lengths=set_lengths,
    )

generated


GeneratedSetlist(sets=[SetSegment(label='Set 1', songs=["Halley's Comet", 'Yarmouth Road', 'Sanity', 'Character Zero', 'Martian Monster', 'Army of One', 'Crowd Control', 'Brian and Robert', 'Devotion To a Dream', 'Water in the Sky']), SetSegment(label='Set 2', songs=['Heavy Things', 'Dirt', '46 Days', 'Set Your Soul Free', 'Simple', 'Theme From the Bottom', 'Backwards Down the Number Line', 'Waves', 'Shine a Light'])], encore=SetSegment(label='Encore', songs=['Loving Cup', 'Tweezer Reprise']), metadata=GenerationMetadata(reference_date=datetime.date(2024, 10, 27), cutoff_date=datetime.date(2018, 12, 31), era='3.0', year=2018, notes=['Excluded 19 songs played on 2024-10-26']))

In [4]:
# Display setlist
def print_segment(segment):
    print(segment.label)
    for idx, song in enumerate(segment.songs, start=1):
        print(f'  {idx}. {song}')
    print()

for segment in generated.sets:
    print_segment(segment)

if generated.encore:
    print_segment(generated.encore)


Set 1
  1. Halley's Comet
  2. Yarmouth Road
  3. Sanity
  4. Character Zero
  5. Martian Monster
  6. Army of One
  7. Crowd Control
  8. Brian and Robert
  9. Devotion To a Dream
  10. Water in the Sky

Set 2
  1. Heavy Things
  2. Dirt
  3. 46 Days
  4. Set Your Soul Free
  5. Simple
  6. Theme From the Bottom
  7. Backwards Down the Number Line
  8. Waves
  9. Shine a Light

Encore
  1. Loving Cup
  2. Tweezer Reprise

